# Taal Lab — Deep Learning Coursework
Run the cells in order. Upload `taal-lab-project.zip` from this project. These experiments use a small teaching corpus, not professional performance data. GPU is optional. The notebook trains Keras models independently of the dependency-free studio.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
target = Path("/content/taal-lab").resolve()
target.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.infolist():
        destination = (target / member.filename).resolve()
        if not destination.is_relative_to(target):
            raise ValueError("Unsafe archive path")
    bundle.extractall(target)
%cd /content/taal-lab

In [ ]:
%pip install -q -r requirements-keras.txt
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

## Unit I: inspect and check backpropagation
The explicit Python RNN exposes every local derivative. Tests compare analytical gradients to central finite differences.

In [ ]:
!python -m unittest discover -s tests -p "test_core.py" -v

## Units II–III: initialization, regularization, optimizers and deep MLP
Compare runs using the same data split. Record validation loss and runtime. Each command writes a model and JSON metrics in `artifacts/experiments/`.

In [ ]:
!python -m coursework.keras_experiments --task mlp --activation relu --batchnorm --dropout 0.2 --optimizer adam --epochs 30
!python -m coursework.optimizer_landscape
from IPython.display import Image, display
display(Image("artifacts/experiments/optimizer_landscape.png"))

In [ ]:
!python -m coursework.keras_experiments --task autoencoder --epochs 20
!python -m coursework.keras_experiments --task cbow --epochs 20
!python -m coursework.keras_experiments --task skipgram --epochs 20

## Unit IV: five MNIST baselines and hyperparameter tuning
The MNIST run downloads the dataset, trains softmax/sigmoid/ReLU/batchnorm/dropout variants and records test accuracy. Tuning uses validation data only. These cells take longer.

In [ ]:
!python -m coursework.keras_experiments --task mnist --epochs 5
!python -m coursework.keras_experiments --task tune --epochs 15

## Unit V: conditional many-to-many LSTM and tabla generation

In [ ]:
!python -m coursework.keras_experiments --task lstm --epochs 35
!python -m coursework.keras_experiments --task generate --model lstm --taal teentaal --cycles 2 --temperature 0.8
from IPython.display import Audio, display
display(Audio("artifacts/experiments/generated.wav"))

## Character RNN with ABC notation
Train on complete-phrase splits before making overlapping windows. Sampled text may contain invalid ABC syntax; inspect it rather than claiming it is valid music.

In [ ]:
!python -m coursework.keras_experiments --task char --epochs 20
!python -m coursework.keras_experiments --task char-generate
!python -m coursework.keras_experiments --task stateful-demo --epochs 3

## Save your results
Download checkpoints, curves and audio for your report. Do not invent metrics for experiments you did not run.

In [ ]:
import shutil
shutil.make_archive("taal-lab-results", "zip", "artifacts/experiments")
files.download("taal-lab-results.zip")